# Projeção do consumo das famílias

Este notebook organiza, de forma reprodutível e didática, a projeção do componente **Despesa de consumo das famílias** do PIB. A lógica aqui é deliberadamente separada do `makedataset.py`: o pipeline principal coleta e trata os dados observados, enquanto este notebook documenta a etapa econométrica, avalia a robustez do modelo e exporta a projeção para `data/processed`.

A estratégia é:

1. Consumir as bases já coletadas pelo `makedataset.py`.
2. Construir uma base trimestral comum.
3. Explorar as séries visualmente e por estatísticas descritivas.
4. Avaliar correlações e uma regressão auxiliar.
5. Estimar um VAR parcimonioso.
6. Testar estabilidade e resíduos.
7. Projetar os próximos trimestres e exportar o resultado para o fluxo do dashboard.

A projeção deve ser lida como uma **trajetória condicional**, não como uma previsão pontual determinística.

## 1. Configuração do ambiente

Carregamos bibliotecas de manipulação de dados, visualização e econometria. O bloco de caminhos foi escrito para funcionar tanto quando o notebook é executado a partir da pasta `src` quanto da raiz do projeto.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

ROOT = Path.cwd()
if ROOT.name.lower() == "src":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"
OUT_PATH = DATA_DIR / "projecao_consumo_familias.parquet"

DATA_DIR

## 2. Consumo dos dados coletados pelo `makedataset.py`

A modelagem não baixa dados diretamente. Ela consome os arquivos processados previamente:

- `pib_componentes_quarterly.parquet`: componentes trimestrais do PIB.
- `ipca_all.parquet`: inflação, incluindo IPCA acumulado em 12 meses.
- `sgs_dados.parquet`: crédito e juros do SGS/BCB.

Essa separação deixa claro o papel de cada etapa: coleta no pipeline, econometria no notebook.

In [ ]:
pib = pd.read_parquet(DATA_DIR / "pib_componentes_quarterly.parquet")
ipca = pd.read_parquet(DATA_DIR / "ipca_all.parquet")
sgs = pd.read_parquet(DATA_DIR / "sgs_dados.parquet")

for df in [pib, ipca, sgs]:
    date_col = "date" if "date" in df.columns else "Date"
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

print("PIB:", pib.columns.tolist())
print("IPCA:", ipca.columns.tolist())
print("SGS:", sgs.columns.tolist())

## 3. Construção da base trimestral de modelagem

O consumo das famílias e a despesa do governo já estão em frequência trimestral, pois vêm das Contas Nacionais Trimestrais. O IPCA e as variáveis de crédito são mensais; por isso, são agregadas para o trimestre pela média.

A variável de crédito usada no VAR é a variação interanual do saldo de crédito PF (`credito_pf_12m`). Essa escolha evita misturar uma série em nível monetário com séries expressas em taxas/variações, reduzindo problemas de escala e interpretação.

In [ ]:
ipca_date_col = "date" if "date" in ipca.columns else "Date"

ipca_q = (
    ipca.set_index(ipca_date_col)[["ipca_12m"]]
    .resample("QE")
    .mean()
    .reset_index()
    .rename(columns={ipca_date_col: "date"})
)

credito_q = (
    sgs.set_index("date")[["taxa_juros_pf", "credito_pf"]]
    .resample("QE")
    .mean()
    .reset_index()
)
credito_q["credito_pf_12m"] = credito_q["credito_pf"].pct_change(4) * 100

base = (
    pib[["date", "consumo_familias", "despesa_governo"]]
    .merge(ipca_q, on="date", how="left")
    .merge(credito_q[["date", "taxa_juros_pf", "credito_pf_12m"]], on="date", how="left")
    .sort_values("date")
    .dropna()
    .reset_index(drop=True)
)

model_cols = ["consumo_familias", "ipca_12m", "despesa_governo", "taxa_juros_pf", "credito_pf_12m"]
base.tail()

## 4. Análise exploratória dos dados

Antes de modelar, examinamos nível, dispersão e comportamento temporal das séries. Como todas as variáveis estão em taxas, variações ou percentuais, a comparação visual é mais informativa do que seria com níveis monetários brutos.

In [ ]:
base[model_cols].describe().T

In [ ]:
fig, axes = plt.subplots(len(model_cols), 1, figsize=(12, 13), sharex=True)

for ax, col in zip(axes, model_cols):
    ax.plot(base["date"], base[col], color="#087f5b", linewidth=2)
    ax.axhline(0, color="#9aa6ac", linewidth=0.8)
    ax.set_title(col)
    ax.set_ylabel("%")

fig.suptitle("Séries trimestrais usadas na projeção", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()

for ax, col in zip(axes, model_cols):
    ax.hist(base[col].dropna(), bins=18, color="#1677a8", alpha=0.82, edgecolor="white")
    ax.set_title(f"Distribuição: {col}")
    ax.set_xlabel("Valor")

axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 5. Correlograma

O correlograma ajuda a identificar relações contemporâneas entre as variáveis. Ele não prova causalidade, mas orienta a especificação: variáveis muito correlacionadas podem trazer multicolinearidade em regressões estáticas e instabilidade em modelos muito parametrizados.

In [ ]:
corr = base[model_cols].corr()

plt.figure(figsize=(9, 7))
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="BrBG", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="#102a35", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title("Correlograma das variáveis do modelo")
plt.show()

## 6. Regressão auxiliar para avaliar parâmetros

Antes do VAR, estimamos uma regressão estática com erro-padrão robusto HAC. Ela não é o modelo final de projeção, mas permite avaliar sinais, magnitude e significância aproximada das variáveis explicativas. Também calculamos VIF para monitorar multicolinearidade.

In [ ]:
y = base["consumo_familias"]
X = base[["ipca_12m", "despesa_governo", "taxa_juros_pf", "credito_pf_12m"]]
X_const = sm.add_constant(X)

ols = sm.OLS(y, X_const).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
print(ols.summary())

vif = pd.DataFrame({
    "variavel": X_const.columns,
    "VIF": [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
})
vif

## 7. Testes de estacionariedade

Como o VAR é sensível a propriedades temporais das séries, usamos dois testes complementares:

- **ADF**: hipótese nula de raiz unitária.
- **KPSS**: hipótese nula de estacionariedade.

A decisão não deve ser mecânica. Em séries macroeconômicas curtas, os testes podem discordar. Aqui, a preferência por taxas e variações interanuais já reduz o risco de trabalhar com níveis claramente não estacionários.

In [ ]:
def stationarity_report(series: pd.Series, name: str) -> dict:
    s = series.dropna()
    adf_stat, adf_p, *_ = adfuller(s, autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")
    return {
        "variavel": name,
        "ADF estatistica": adf_stat,
        "ADF p-valor": adf_p,
        "KPSS estatistica": kpss_stat,
        "KPSS p-valor": kpss_p,
    }

stationarity = pd.DataFrame([stationarity_report(base[col], col) for col in model_cols])
stationarity

## 8. Estimação do VAR

O VAR trata todas as variáveis como endógenas e permite que a dinâmica passada do sistema ajude a projetar o consumo das famílias. Para evitar sobreparametrização, limitamos o número máximo de defasagens e escolhemos a ordem pelo AIC, respeitando um mínimo de uma defasagem.

In [ ]:
var_data = base.set_index("date")[model_cols]

maxlags = max(1, min(4, len(var_data) // 8))
selector = VAR(var_data).select_order(maxlags=maxlags)
print(selector.summary())

lag_order = int(selector.aic) if selector.aic else 1
lag_order = max(1, min(lag_order, maxlags))

var_model = VAR(var_data).fit(lag_order)
print(var_model.summary())
print(f"Defasagens selecionadas: {lag_order}")

## 9. Testes e robustez do VAR

A avaliação de robustez combina três dimensões:

1. **Estabilidade dinâmica**: as raízes do VAR devem ficar fora do círculo unitário na convenção do `statsmodels`, e `is_stable()` deve retornar verdadeiro.
2. **Autocorrelação dos resíduos**: resíduos muito autocorrelacionados indicam que a dinâmica do modelo ainda não foi bem capturada.
3. **Normalidade dos resíduos**: não é condição absoluta para projeção, mas ajuda a interpretar intervalos construídos a partir da variância residual.

Se algum teste for fraco, o resultado deve ser tratado como sinal prospectivo exploratório, não como cenário fechado.

In [ ]:
print("VAR estável?", var_model.is_stable(verbose=True))

print("\nTeste de autocorrelação dos resíduos:")
try:
    print(var_model.test_whiteness(nlags=max(lag_order + 4, 8)).summary())
except Exception as exc:
    print(f"Não foi possível calcular o teste de autocorrelação: {exc}")

print("\nTeste de normalidade dos resíduos:")
try:
    print(var_model.test_normality().summary())
except Exception as exc:
    print(f"Não foi possível calcular o teste de normalidade: {exc}")

resid = var_model.resid
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(resid.index, resid["consumo_familias"], color="#087f5b")
ax.axhline(0, color="#9aa6ac", linewidth=0.8)
ax.set_title("Resíduos do VAR: equação do consumo das famílias")
plt.show()

## 10. Projeção e cenários

Projetamos nove trimestres à frente. O intervalo de cenário é construído a partir do desvio-padrão dos resíduos da equação do consumo, ampliado pela raiz do horizonte. É uma aproximação simples e transparente: quanto mais distante o horizonte, maior a incerteza.

Na exportação, mantemos histórico e projeção no mesmo arquivo:

- `consumo_observado`: valores históricos.
- `consumo_projecao`: trajetória central projetada.
- `consumo_cenario_baixo` e `consumo_cenario_alto`: faixas de incerteza.

In [ ]:
steps = 9
last_date = var_data.index[-1]
forecast_dates = pd.date_range(last_date + pd.offsets.QuarterEnd(), periods=steps, freq="QE")

forecast_values = var_model.forecast(var_data.values[-lag_order:], steps=steps)
forecast_df = pd.DataFrame(forecast_values, columns=model_cols, index=forecast_dates).reset_index(names="date")

resid_std = float(var_model.resid["consumo_familias"].std())
scale = np.sqrt(np.arange(1, steps + 1))

forecast_df["tipo"] = "projecao"
forecast_df["metodo"] = f"VAR({lag_order})"
forecast_df["consumo_observado"] = np.nan
forecast_df["consumo_projecao"] = forecast_df["consumo_familias"]
forecast_df["consumo_cenario_baixo"] = forecast_df["consumo_projecao"] - resid_std * scale
forecast_df["consumo_cenario_alto"] = forecast_df["consumo_projecao"] + resid_std * scale

hist = base[["date", *model_cols]].copy()
hist["tipo"] = "observado"
hist["metodo"] = f"VAR({lag_order})"
hist["consumo_observado"] = hist["consumo_familias"]
hist["consumo_projecao"] = np.nan
hist["consumo_cenario_baixo"] = np.nan
hist["consumo_cenario_alto"] = np.nan

output_cols = [
    "date", "tipo", "metodo",
    "consumo_observado", "consumo_projecao", "consumo_cenario_baixo", "consumo_cenario_alto",
    "ipca_12m", "despesa_governo", "taxa_juros_pf", "credito_pf_12m",
]

projecao = pd.concat([hist, forecast_df], ignore_index=True, sort=False)[output_cols]
projecao.tail(12)

In [ ]:
plot_df = projecao.tail(48)

plt.figure(figsize=(12, 6))
plt.plot(plot_df["date"], plot_df["consumo_observado"], label="Observado", color="#087f5b", linewidth=2.5)
plt.plot(plot_df["date"], plot_df["consumo_projecao"], label="Projeção central", color="#1677a8", linestyle="--", linewidth=2.5)
plt.fill_between(
    plot_df["date"],
    plot_df["consumo_cenario_baixo"].astype(float),
    plot_df["consumo_cenario_alto"].astype(float),
    color="#b6ddd1",
    alpha=0.35,
    label="Faixa de cenário",
)
plt.axvline(forecast_dates[0], color="#9aa6ac", linestyle=":", linewidth=1.2)
plt.title("Projeção do consumo das famílias")
plt.ylabel("Variação trimestral / interanual conforme base do PIB (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Exportação para o fluxo do projeto

A saída final é gravada em `data/processed/projecao_consumo_familias.parquet`. Depois disso, basta rodar `dashboard/scripts/export_data.py` para atualizar o JSON usado pelo Next.js.

In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
projecao.to_parquet(OUT_PATH, index=False)
print(f"Arquivo exportado: {OUT_PATH}")
print(f"Linhas exportadas: {len(projecao)}")